In [10]:
# Install HuggingFace libraries - Colab may not have latest versions
!pip install transformers datasets accelerate -q

In [11]:
# Import datasets library to load ready-made datasets from HuggingFace
from datasets import load_dataset

# Download SMS Spam Collection - 5,574 real labeled SMS messages
dataset = load_dataset("sms_spam")

# Print structure - shows features and number of rows
print(dataset)

# Print first message - one dictionary with sms text and label
print(dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})
{'sms': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n', 'label': 0}


In [12]:
# Split full dataset: 80% for training, 20% for testing
# test_size=0.2 means 20% goes to test
dataset = dataset["train"].train_test_split(test_size=0.2)

# Should show train: 4459, test: 1115
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 4459
    })
    test: Dataset({
        features: ['sms', 'label'],
        num_rows: 1115
    })
})


In [13]:
# AutoTokenizer loads correct tokenizer for any HuggingFace model
from transformers import AutoTokenizer

# Load DistilBERT tokenizer - converts text to numbers
# uncased = treats uppercase and lowercase as same
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Test on one sentence to see how it works
result = tokenizer("Congratulations! You won a free iPhone.")
print(result)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'input_ids': [101, 23156, 999, 2017, 2180, 1037, 2489, 18059, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [14]:
# Define function that tokenizes each message
def tokenize_function(example):
    return tokenizer(
        example["sms"],           # the text column from each row
        truncation=True,          # cut messages longer than 128 tokens
        padding="max_length",     # pad all messages to exactly 128 tokens
        max_length=128            # fixed length - all messages same size
    )

# Apply to ALL 5,574 messages at once using .map()
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Remove raw text column - Trainer only needs numbers
tokenized_dataset = tokenized_dataset.remove_columns(["sms"])

# Convert to PyTorch tensors - required for training
tokenized_dataset.set_format("torch")

# Should show input_ids, attention_mask, token_type_ids, label columns
print(tokenized_dataset)

Map:   0%|          | 0/4459 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4459
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1115
    })
})


In [15]:
# AutoModelForSequenceClassification loads DistilBERT + classification head
from transformers import AutoModelForSequenceClassification

# Load DistilBERT with classification head on top
# num_labels=2 because we have 2 categories: spam(1) and not spam(0)
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print(model)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [16]:
from transformers import TrainingArguments, Trainer
import numpy as np

# Define how we measure model performance after each epoch
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Take highest scoring label as final prediction
    predictions = np.argmax(predictions, axis=1)
    # Calculate accuracy: correct / total
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

# Configure training process
training_args = TrainingArguments(
    output_dir="./spam_model",          # where to save model
    num_train_epochs=3,                 # pass through all data 3 times
    per_device_train_batch_size=16,     # process 16 messages at a time
    per_device_eval_batch_size=16,      # evaluate 16 messages at a time
    eval_strategy="epoch",              # check accuracy after each epoch
    save_strategy="no",                 # don't save to disk
    logging_steps=10,                   # print progress every 10 steps
)

# Build Trainer - connects everything together
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],   # 4,459 messages for learning
    eval_dataset=tokenized_dataset["test"],     # 1,115 messages for testing
    compute_metrics=compute_metrics,
)

print("Trainer ready")

Trainer ready


In [17]:
# Start fine-tuning
# On Colab GPU this takes 10-15 minutes
# Will show progress bar + accuracy after each epoch
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.041805,0.039453,0.992825
2,0.019602,0.029179,0.994619
3,0.014443,0.033885,0.994619


TrainOutput(global_step=837, training_loss=0.03537466616220683, metrics={'train_runtime': 159.7445, 'train_samples_per_second': 83.74, 'train_steps_per_second': 5.24, 'total_flos': 443004097955328.0, 'train_loss': 0.03537466616220683, 'epoch': 3.0})

In [18]:
# Import pipeline tool from HuggingFace
from transformers import pipeline

# Wrap your trained model into a pipeline for easy use
# "text-classification" tells it what task this model does
spam_detector = pipeline("text-classification", model=model, tokenizer=tokenizer)
# A list of 6 test messages — mix of spam and normal
test_messages = [
    "Congratulations! You won a free iPhone. Click now!",
    "Hey, are we meeting tomorrow?",
    "URGENT: Your bank account is compromised. Call now!",
    "Can you send me the project files?",
    "FREE MONEY! Claim your prize today!",
    "What time is lunch?"
]
# Loop through each message one by one
for message in test_messages:

    # Run the message through the spam detector
    result = spam_detector(message)[0]

    # Model returns LABEL_1 = spam, LABEL_0 = not spam
    # This line converts that into plain English
    label = "SPAM" if result["label"] == "LABEL_1" else "NOT SPAM"

    # Convert confidence score from 0-1 to percentage
    confidence = round(result["score"] * 100, 2)

    # Print: SPAM (97.3%) → Congratulations! You won...
    print(f"{label} ({confidence}%) → {message}")

SPAM (99.96%) → Congratulations! You won a free iPhone. Click now!
NOT SPAM (99.98%) → Hey, are we meeting tomorrow?
NOT SPAM (94.6%) → URGENT: Your bank account is compromised. Call now!
NOT SPAM (99.95%) → Can you send me the project files?
SPAM (99.95%) → FREE MONEY! Claim your prize today!
NOT SPAM (99.96%) → What time is lunch?


In [19]:
# Install HuggingFace Hub library
!pip install huggingface_hub -q

In [20]:
# Login to HuggingFace using your token
from huggingface_hub import login
login()

In [21]:
from huggingface_hub import login
from google.colab import userdata

# Get token from Colab secrets (secure way)
hf_token = userdata.get('HF_TOKEN')

# Login to HuggingFace
login(hf_token)
print("Login successful!")

Login successful!


In [22]:
# Push trained model and tokenizer to HuggingFace Hub
model.push_to_hub("Danisht/spam-detector-distilbert")
tokenizer.push_to_hub("Danisht/spam-detector-distilbert")
print("Model pushed successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...eg3acvr/model.safetensors:   2%|2         | 6.43MB /  268MB            

README.md: 0.00B [00:00, ?B/s]

Model pushed successfully!


In [24]:
from huggingface_hub import ModelCard

card_content = """
# Spam Detector — DistilBERT

## Model Description
A fine-tuned DistilBERT model for SMS spam detection. Classifies messages as SPAM or NOT SPAM.

- **Developed by:** Muhammad Danisht
- **Model type:** Text Classification
- **Language:** English
- **Finetuned from:** distilbert-base-uncased

## Training Data
- Dataset: SMS Spam Collection (5,574 messages)
- Split: 80% train / 20% test

## Training Results
| Epoch | Accuracy |
|-------|----------|
| 1 | 98.92% |
| 2 | 98.92% |
| 3 | 99.19% |

## How to Use
```python
from transformers import pipeline
spam_detector = pipeline("text-classification", model="Danisht/spam-detector-distilbert")
result = spam_detector("Congratulations! You won a free iPhone!")
print(result)
```

## Limitations
Model trained on 2011 SMS data. May miss modern phishing patterns.

## GitHub
https://github.com/muhammeddanisht
"""

card = ModelCard(card_content)
card.push_to_hub("Danisht/spam-detector-distilbert")
print("Model card updated!")

Repo card metadata block was not found. Setting CardData to empty.


Model card updated!
